# Zalo AI Challenge 2026 — LifeMate (ReAct + Schema-Grounded QLoRA)

Thin-runner notebook cho Google Colab (T4 16GB):
- **Cell 1–2:** Mount Drive, đồng bộ `duylhm/ZaloAIC-Lifemate`, cài thư viện, giải nén dữ liệu & chọn model `<5B`
- **Cell 3–4 *(Tùy chọn)*:** Benchmark ASR trên audio `train` & đo điểm zero-shot
- **Cell 5 *(Khuyên chạy)*:** Fine-tune QLoRA trên `train_ids` (`1,303`) & chấm Unseen-App CV (`142` unseen + `100` in-domain)
- **Cell 6–7 *(Bắt buộc)*:** Fine-tune Full (`1,545` mẫu) → Infer `public_test` (`634`) → Validate & Tải `submission.jsonl` về máy

In [ ]:
import os, pathlib, subprocess

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = pathlib.Path("/content/ZaloAIC-Lifemate")
if not (REPO_DIR / ".git").exists():
    !rm -rf /content/ZaloAIC-Lifemate
    !git clone https://github.com/duylhm/ZaloAIC-Lifemate.git /content/ZaloAIC-Lifemate
else:
    !git -C /content/ZaloAIC-Lifemate pull --ff-only

%cd /content/ZaloAIC-Lifemate
!pip install -q -U -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

DATA_DIR = pathlib.Path("/content/data")
has_train = (DATA_DIR / "train").exists() or (DATA_DIR / "train+public_test" / "train").exists()
if not has_train:
    zip_candidates = [
        pathlib.Path("/content/drive/MyDrive/train+public_test.zip"),
        *pathlib.Path("/content/drive/MyDrive").rglob("train+public_test.zip"),
    ]
    zip_path = next((p for p in zip_candidates if p.is_file()), None)
    if zip_path is None:
        raise FileNotFoundError("Không tìm thấy train+public_test.zip trên Google Drive. Hãy upload vào MyDrive rồi chạy lại Cell 1.")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f"[Unzip] {zip_path} -> {DATA_DIR}")
    !unzip -q -o "{zip_path}" -d /content/data

!python scripts/prepare_splits.py

In [ ]:
# Chọn base model <5B params: "Qwen/Qwen3-4B" (tắt thinking) hoặc "Qwen/Qwen2.5-3B-Instruct"
MODEL_ID = "Qwen/Qwen3-4B"
MODEL_TAG = MODEL_ID.split("/")[-1]
CKPT_ROOT = f"/content/drive/MyDrive/lifemate_checkpoints/{MODEL_TAG}"
print(f"Active MODEL_ID={MODEL_ID} | Checkpoints={CKPT_ROOT}")

In [ ]:
# [Tùy chọn] Đo CER/WER tiếng Việt trên 50 file .wav tập train
!python scripts/benchmark_asr.py --models vinai/PhoWhisper-large openai/whisper-large-v3-turbo --limit 50 --num-beams 3

In [ ]:
# [Tùy chọn] Đo mốc Zero-shot trên 242 mẫu val_all trước khi fine-tune
!python run_inference.py \
  --split train \
  --ids-file splits/val_all.json \
  --model "{MODEL_ID}" \
  --force \
  --output outputs/val_zeroshot.jsonl \
  --eval

In [ ]:
# Fine-tune QLoRA trên train_ids (1,303) & chấm điểm Unseen-App CV (142 unseen + 100 in-domain)
VAL_ADAPTER = f"{CKPT_ROOT}_val"
!python train_sft.py \
  --model "{MODEL_ID}" \
  --split-mode val \
  --epochs 2 \
  --aug-copies 1 \
  --max-seq-len 6144 \
  --output-dir "{VAL_ADAPTER}"

!python run_inference.py \
  --split train \
  --ids-file splits/val_all.json \
  --model "{MODEL_ID}" \
  --adapter "{VAL_ADAPTER}" \
  --force \
  --output outputs/val_sft.jsonl \
  --eval

!python scripts/inspect_errors.py \
  --split train \
  --ids-file splits/val_all.json \
  --submission outputs/val_sft.jsonl \
  --max-per-bucket 5

In [ ]:
# Fine-tune Full 1,545 mẫu -> Infer public_test (634) -> Validate & lưu Drive
FULL_ADAPTER = f"{CKPT_ROOT}_full"
!python train_sft.py \
  --model "{MODEL_ID}" \
  --split-mode full \
  --epochs 2 \
  --aug-copies 1 \
  --max-seq-len 6144 \
  --output-dir "{FULL_ADAPTER}"

!python run_inference.py \
  --split public_test \
  --model "{MODEL_ID}" \
  --adapter "{FULL_ADAPTER}" \
  --force \
  --output /content/submission.jsonl \
  --mirror-path /content/drive/MyDrive/submission.jsonl

import pathlib
sub_p = pathlib.Path("/content/submission.jsonl")
print(f"[Verify] Lines={len(sub_p.read_text(encoding='utf-8').splitlines())} | Backup=/content/drive/MyDrive/submission.jsonl")
try:
    from google.colab import files
    files.download(str(sub_p))
except Exception as exc:
    print(f"Tự động tải trình duyệt bị chặn ({exc}). Chạy Cell 7 bên dưới hoặc tải từ Google Drive MyDrive/submission.jsonl.")

In [ ]:
# [Tải lại thủ công] Dùng khi trình duyệt chặn popup download tự động ở Cell 6
import pathlib, shutil
from google.colab import files

sub_p = pathlib.Path("/content/submission.jsonl")
drive_p = pathlib.Path("/content/drive/MyDrive/submission.jsonl")
target = sub_p if sub_p.is_file() else drive_p
assert target.is_file(), "Chưa có submission.jsonl. Hãy chạy Cell 6 trước."
print(f"Downloading {target} ({len(target.read_text(encoding='utf-8').splitlines())} lines)...")
files.download(str(target))